# De una pregunta social a un grafo consultable
**Taller FACSO · Universidad Alberto Hurtado · 30 de octubre de 2026**  
**Caso:** Santuario del Señor de Qoyllur Rit’i · **Formato:** Google Colab

## ¿Qué haremos?

En Ciencias Sociales, un **grafo** permite representar entidades (personas, lugares, instituciones, acontecimientos) y las **relaciones** entre ellas. Comenzaremos con solo cinco entidades y cuatro relaciones que proceden de un archivo real del proyecto Qoyllur Rit’i.

Primero veremos qué dice el grafo. Después consultaremos **Wikidata**, una base de conocimiento abierta en la que las entidades tienen identificadores `Q` y las relaciones se expresan mediante propiedades `P`. Para consultarla utilizaremos **SPARQL**, un lenguaje que busca patrones de relaciones.

Finalmente probaremos un **mini-RAG**: ante una pregunta, el sistema recupera primero relaciones del grafo, presenta las evidencias y, opcionalmente, pide a un modelo de lenguaje que redacte una respuesta basada en ellas.

### Cinco operaciones
1. Leer nodos y relaciones (no hace falta escribir código).
2. Visualizar y examinar sus metadatos, como tipos y etiquetas.
3. Consultar relaciones de una entidad en Wikidata.
4. Recuperar las relaciones pertinentes a una pregunta.
5. Opcionalmente generar una respuesta con un modelo pequeño, **verificando la evidencia**.

**Para empezar:** en Colab elegir **Entorno de ejecución → Ejecutar todas**. El ejemplo ya está incluido; no necesitas subir nada. La generación opcional con modelo está desactivada para que la sesión sea reproducible. Si deseas incorporar tu propio grafo, ejecuta la celda **«Subir otro grafo»** y vuelve a correr las etapas posteriores.

**Formatos admitidos:** `.ttl`, `.rdf`, `.nt`, `.jsonld`, `.graphml`, `.gexf` y CSV con columnas `origen,destino,tipo`.

**Procedencia:** los identificadores `example.org` pertenecen a nuestro grafo local; no son enlaces públicos ni identificadores de Wikidata. No publiques datos sensibles sin una revisión específica.


## 1. Preparación
Instalamos herramientas abiertas: `rdflib` para RDF, `networkx` para grafos y `scikit-learn` para la recuperación de relaciones. No se necesita cuenta de pago ni clave API.

In [ ]:
%pip -q install "rdflib>=7,<9" "networkx>=3,<5" "scikit-learn>=1.2,<2" pandas matplotlib requests

In [ ]:
from pathlib import Path
import re, textwrap
from collections import Counter
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from rdflib import Graph as RDFGraph, URIRef, Literal, RDF, RDFS, OWL
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 100)
print("Bibliotecas cargadas.")

## 2. Leer nuestro grafo: ¿qué significa TTL?

Un archivo **`.ttl`** está escrito en **Turtle**, una forma legible de almacenar datos **RDF**. En RDF, una afirmación se escribe como un **triple**: **sujeto → propiedad → objeto**. Por ejemplo, `SantuarioQoylluriti → estaEn → Sinakara`.

El subgrafo de esta práctica contiene **cinco entidades y cuatro relaciones explícitas** del TTL aportado. Es una selección didáctica, no todo el grafo de Qoyllur Rit’i.

| Relación | Qué representa |
|---|---|
| Festividad → ocurreEnLugar → Santuario | Lugar vinculado al desarrollo de la festividad |
| Santuario → estaEn → Sinakara | Ubicación del santuario en el modelo |
| Sinakara → estaEn → Ausangate | Relación territorial consignada |
| Santuario → contiene → Mamachapata | Un lugar dentro del recinto |

En RDF, `rdf:type` indica el **tipo** de una entidad; `rdfs:label` es el **nombre legible** y `rdfs:comment` proporciona una **descripción**. Son metadatos importantes, pero para el dibujo de esta práctica nos interesan sobre todo las cuatro relaciones entre las cinco entidades.


In [ ]:
EJEMPLO_TTL = '@prefix fest: <http://example.org/festividades#> .\n@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .\n\nfest:FestividadQoylluriti a fest:Festividad ;\n    rdfs:label "Festividad del Señor de Qoyllur Rit\'i"@es ;\n    fest:ocurreEnLugar fest:SantuarioQoylluriti ;\n    fest:tieneImportancia "central"@es ;\n    rdfs:comment "Peregrinación andina de carácter ritual que se realiza anualmente en el área de Sinakara en honor al Señor de Qoyllur Rit\'i. La festividad tiene una duración aproximada de 5 días (120 horas), comenzando con la llegada de peregrinos y culminando con el retorno a sus comunidades. Es una de las festividades religiosas más importantes de los Andes peruanos, congregando a decenas de miles de peregrinos de múltiples naciones. La festividad sincretiza elementos del catolicismo colonial con tradiciones andinas ancestrales, siendo el culto al Apu Ausangate un eje fundamental. Incluye eventos clave como la subida ritual al glaciar Colque Punku, la lomada (caminata nocturna de 24 horas), procesiones, misas especiales y complejos sistemas de intercambio simbólico entre comunidades participantes."@es .\n\nfest:Ausangate a fest:Apu ;\n    fest:tieneImportancia "central"@es ;\n    rdfs:comment "Apu Ausangate es el espíritu protector de la montaña más alta de la región Cusco (6384 msnm). En la cosmovisión andina, es uno de los apus más importantes. El nevado Ausangate contiene el glaciar Colque Punku, donde se realizan rituales durante Qoyllur Rit\'i."@es .\n\nfest:Mamachapata a fest:Lugar ;\n    rdfs:label "Mamachapata"@es ;\n    fest:tieneImportancia "central"@es ;\n    rdfs:comment "Área dentro del recinto del santuario hacia donde se dirige la procesión con la imagen del Señor de Tayankani."@es .\n\nfest:SantuarioQoylluriti a fest:Lugar,\n        fest:LugarSagrado ;\n    rdfs:label "Santuario del Señor de Qoyllur Rit\'i"@es ;\n    fest:contiene fest:Mamachapata ;\n    fest:estaEn fest:Sinakara ;\n    fest:tieneImportancia "central"@es ;\n    rdfs:comment "Santuario de alta montaña donde se concentra la peregrinación anual del Señor de Qoyllur Rit’i."@es .\n\nfest:Sinakara a fest:Lugar,\n        fest:LugarSagrado ;\n    rdfs:label "Sinakara"@es ;\n    fest:estaEn fest:Ausangate ;\n    fest:tieneAltitudMetros 4650 ;\n    fest:tieneImportancia "central"@es ;\n    rdfs:comment "Área montañosa sagrada ubicada en las faldas del nevado Ausangate, donde se encuentra el Santuario del Señor de Qoyllur Rit\'i. Sinakara es el centro geográfico de la peregrinación anual y concentra los eventos principales de la festividad."@es .\n\n'

NOMBRES_RELACIONES = {
    "ocurreEnLugar": "ocurre en el lugar",
    "estaEn": "está en",
    "contiene": "contiene",
    "realizadoPor": "realizado por",
    "esParteDe": "es parte de",
    "tieneParte": "tiene como parte",
    "participaEn": "participa en",
}

def nombre_corto(iri):
    return str(iri).rsplit("#",1)[-1].rsplit("/",1)[-1]

def nombre_relacion(iri):
    s=nombre_corto(iri)
    return NOMBRES_RELACIONES.get(s, re.sub(r"(?<=[a-záéíóú])(?=[A-Z])", " ", s).replace("_", " ").lower())

PROPIEDADES_NO_RELACIONALES = {RDF.type,RDFS.label,RDFS.comment,RDFS.subClassOf,RDFS.domain,RDFS.range}
NS_PROHIBIDOS = (str(RDF),str(RDFS),str(OWL), "http://www.w3.org/2002/07/owl#")

def desde_rdf(grafo_rdf, fuente="archivo RDF"):
    # Entidades nombradas; las declaraciones de clases y propiedades no son individuos.
    entidades={s for s in grafo_rdf.subjects(RDF.type,OWL.NamedIndividual) if isinstance(s, URIRef)}
    if not entidades:
        clases_propiedades={s for tipo in [OWL.Class,OWL.ObjectProperty,OWL.DatatypeProperty,RDFS.Class,RDF.Property] for s in grafo_rdf.subjects(RDF.type,tipo)}
        entidades={s for s,p,o in grafo_rdf if isinstance(s,URIRef) and (isinstance(o,Literal) or isinstance(o,URIRef)) and s not in clases_propiedades}
    red=nx.MultiDiGraph()
    for s in entidades:
        eti=next((str(x) for x in grafo_rdf.objects(s,RDFS.label) if isinstance(x,Literal)),nombre_corto(s))
        tipos=sorted({nombre_corto(x) for x in grafo_rdf.objects(s,RDF.type) if x!=OWL.NamedIndividual and isinstance(x,URIRef)})
        descr=next((str(x) for x in grafo_rdf.objects(s,RDFS.comment) if isinstance(x,Literal)),"")
        # Las propiedades literales se conservan como metadatos de cada nodo.
        attrs={nombre_corto(p):str(o) for p,o in grafo_rdf.predicate_objects(s) if isinstance(o,Literal) and p not in [RDFS.label,RDFS.comment]}
        red.add_node(str(s),etiqueta=eti,tipo=", ".join(tipos) or "no especificado",descripcion=descr,metadatos=attrs,fuente=fuente)
    for s,p,o in grafo_rdf:
        if s not in entidades or o not in entidades or p in PROPIEDADES_NO_RELACIONALES:continue
        if str(p).startswith(NS_PROHIBIDOS):continue
        red.add_edge(str(s),str(o),relacion=nombre_relacion(p),propiedad=nombre_corto(p),iri_propiedad=str(p),fuente=fuente)
    if red.number_of_edges()==0:
        raise ValueError("No hay relaciones de entidad a entidad. Comprueba que tu archivo incluye sujetos y objetos URI y relaciones entre ellos.")
    return red

def cargar_grafo(ruta):
    ruta=Path(ruta)
    ext=ruta.suffix.lower()
    if ext in {".ttl",".rdf",".nt",".jsonld"}:
        format_map={".ttl":"turtle",".rdf":"xml",".nt":"nt",".jsonld":"json-ld"}
        grafo_rdf=RDFGraph().parse(str(ruta),format=format_map[ext]);
        print(f"Archivo RDF: {len(grafo_rdf)} triples (incluye tipos, descripciones y esquema).")
        return desde_rdf(grafo_rdf, fuente=ruta.name)
    if ext in {".graphml",".gexf"}:
        base=nx.read_graphml(ruta) if ext==".graphml" else nx.read_gexf(ruta)
        red=nx.MultiDiGraph(base)
        for n,attr in red.nodes(data=True):
            attr.setdefault('etiqueta',str(attr.get('label',n)))
            attr.setdefault('tipo',str(attr.get('tipo','no especificado')))
            attr.setdefault('fuente',ruta.name)
        for s,o,clave,attr in red.edges(keys=True,data=True):
            attr.setdefault('relacion',str(attr.get('tipo',attr.get('label','relacionado con'))))
            attr.setdefault('fuente',ruta.name)
        return red
    if ext==".csv":
        datos=pd.read_csv(ruta)
        req={'origen','destino','tipo'}
        if not req.issubset(datos.columns):raise ValueError('El CSV debe tener las columnas origen, destino, tipo. Opcionales: fuente, evidencia.')
        red=nx.MultiDiGraph()
        for i,fila in datos.iterrows():
            a,b=str(fila['origen']).strip(),str(fila['destino']).strip()
            if not a or not b or a=='nan' or b=='nan':continue
            red.add_node(a,etiqueta=a,fuente=ruta.name)
            red.add_node(b,etiqueta=b,fuente=ruta.name)
            red.add_edge(a,b,relacion=str(fila['tipo']),fuente=str(fila.get('fuente',ruta.name)),evidencia=str(fila.get('evidencia','')))
        # Si existe un CSV adicional de nodos, conservar etiquetas, tipos y descripción.
        for posible in ruta.parent.glob('*nodos*.csv'):
            try:
                metadatos=pd.read_csv(posible)
                if {'id','etiqueta'}.issubset(metadatos.columns):
                    for _,nodo in metadatos.iterrows():
                        identificador=str(nodo['id'])
                        if identificador in red.nodes:
                            red.nodes[identificador].update({campo:str(nodo[campo]) for campo in ['etiqueta','tipo','descripcion','fuente'] if campo in metadatos.columns and pd.notna(nodo[campo])})
            except Exception as error:
                print('Aviso: no se pudieron leer metadatos de',posible,':',error)
        return red
    raise ValueError('Formato no compatible: '+ext)

# Se puede comenzar inmediatamente con el ejemplo, sin descargar un archivo.
grafo_de_prueba=RDFGraph().parse(data=EJEMPLO_TTL,format='turtle')
G=desde_rdf(grafo_de_prueba,fuente='santuario_ejemplo.ttl (selección del TTL local)')
print(f'Grafo inicial: {G.number_of_nodes()} entidades y {G.number_of_edges()} relaciones.')

### Subir otro grafo (opcional)
Para trabajar con tu archivo, ejecuta **solo esta celda**, selecciona un archivo y después vuelve a ejecutar la visualización y las consultas. La subida es a la sesión de Colab, no a GitHub. Usa únicamente datos que puedas compartir en ese entorno.

**Importante:** en archivos RDF/Turtle la visualización utiliza entidades y relaciones entre entidades. Las declaraciones de clases y propiedades quedan fuera de la red dibujada; los atributos literales se conservan como metadatos.

In [ ]:
SUBIR_ARCHIVO_AHORA = False  # Cambia a True y ejecuta esta celda para subir un archivo.

if SUBIR_ARCHIVO_AHORA:
    try:
        from google.colab import files as archivos_colab
        print("Elige un archivo .ttl, .rdf, .nt, .jsonld, .gexf, .graphml o .csv")
        archivos_subidos=archivos_colab.upload()
        if archivos_subidos:
            posibles=list(archivos_subidos)
            compatibles=['.ttl','.rdf','.nt','.jsonld','.graphml','.gexf','.csv']
            nombres=[x for x in posibles if Path(x).suffix.lower() in compatibles and not ('nodos' in x.lower() and x.lower().endswith('.csv'))]
            if not nombres:
                print('No se encontró un archivo de aristas o grafo. Puedes elegir un TTL o un CSV con origen,destino,tipo.')
            else:
                nombre_archivo=sorted(nombres,key=lambda x:compatibles.index(Path(x).suffix.lower()))[0]
                G=cargar_grafo(nombre_archivo)
                print(f"Grafo cargado: {G.number_of_nodes()} entidades y {G.number_of_edges()} relaciones.")
        else:
            print("Sin archivo nuevo: se conserva el grafo anterior.")
    except ImportError:
        print("En Jupyter local usa: G=cargar_grafo('ruta/al/archivo.ttl')")
else:
    print("Se conserva el grafo de ejemplo. Para subir otro, cambia SUBIR_ARCHIVO_AHORA a True y ejecuta esta celda.")

## 3. Metadatos y estructura
Un grafo tiene **vértices** y **aristas**, pero además puede conservar **tipo**, **etiqueta**, **fuente** y **propiedades**. La etiqueta es para leer; el identificador es para integrar y reproducir.

In [ ]:
df_nodos=pd.DataFrame([{"id":nombre_corto(n),"etiqueta":d.get('etiqueta',nombre_corto(n)),"tipo":d.get('tipo',''),"fuente":d.get('fuente','')} for n,d in G.nodes(data=True)])
df_aristas=pd.DataFrame([{"origen":G.nodes[a].get('etiqueta',nombre_corto(a)),"relación":d.get('relacion',''),"destino":G.nodes[b].get('etiqueta',nombre_corto(b)),"fuente":d.get('fuente','')} for a,b,d in G.edges(data=True)])
print('Entidades:'); display(df_nodos.head(20))
print('Relaciones:'); display(df_aristas.head(30))
print('Totales:',G.number_of_nodes(),'nodos; ',G.number_of_edges(),'aristas')

# Centralidad de grado en la red sin dirección; es solo descriptiva para este ejemplo.
red_simple=nx.Graph(G)
centralidad=nx.degree_centrality(red_simple)
for n,puntaje in sorted(centralidad.items(),key=lambda t:-t[1])[:8]:
    print(f"  {G.nodes[n].get('etiqueta',nombre_corto(n))}: {puntaje:.2f}")

In [ ]:
if G.number_of_nodes()>50:
    print('El grafo es grande: se dibujan solo 50 entidades para mantener la lectura.')
    seleccion=sorted(G.degree(),key=lambda x:-x[1])[:50]
    vista=G.subgraph([n for n,_ in seleccion])
else:
    vista=G
red_vista=nx.Graph(vista)
plt.figure(figsize=(12,7))
pos=nx.spring_layout(red_vista,seed=16)
nx.draw_networkx_nodes(red_vista,pos,node_size=1250,node_color='#d5eae5',edgecolors='#116e67',linewidths=1.8)
nx.draw_networkx_edges(red_vista,pos,arrows=False,edge_color='#7da39c',width=2)
nx.draw_networkx_labels(red_vista,pos,labels={n:textwrap.fill(vista.nodes[n].get('etiqueta',nombre_corto(n)),18) for n in vista.nodes},font_size=8)
plt.axis('off');plt.title('Grafo construido a partir de relaciones documentadas',fontsize=14)
plt.show()

## 4. Wikidata y SPARQL: una segunda fuente

**Wikidata** es una base de conocimiento abierta que registra afirmaciones estructuradas. No es lo mismo que Wikipedia: en lugar de artículos en prosa, vincula elementos mediante propiedades.

- Un elemento tiene un identificador que empieza por **Q**. La **festividad** Qoyllur Rit’i es [Q2408955](https://www.wikidata.org/wiki/Q2408955).
- Una propiedad empieza por **P**: `P17` significa **país**, `P131` **entidad territorial** y `P31` **instancia de** (tipo).
- **SPARQL** es el lenguaje de consultas que nos permite preguntar qué valores están conectados con una entidad a través de ciertas propiedades.

**Importante:** el Santuario de nuestro TTL local es un **lugar** y Q2408955 es la **festividad**. No fusionamos ambos nodos porque sus nombres estén relacionados.

La consulta se ejecuta en [Wikidata Query Service](https://query.wikidata.org/). Si falla, el cuaderno presenta un **respaldo identificado** verificado el 29 de octubre de 2026; nunca debe confundirse con un resultado obtenido en directo.

Wikidata puede tener errores o información ausente. La falta de una relación en la base no prueba su ausencia en la práctica cultural.


### Anatomía de una consulta SPARQL y ejemplos reutilizables

**SPARQL consulta patrones de relaciones**, a diferencia del mini-RAG que ordena frases según su similitud textual.

```sparql
SELECT ?pais ?paisLabel WHERE {
  wd:Q2408955 wdt:P17 ?pais.
  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "es,en".
  }
}
LIMIT 10
```

- `SELECT`: columnas que queremos recuperar; `?pais` es la variable para el objeto.
- `WHERE`: condiciones de los triples.
- `wd:Q2408955`: festividad de Qoyllur Rit’i, una **entidad**.
- `wdt:P17`: propiedad directa «país»; el punto cierra el triple.
- `SERVICE wikibase:label`: pide etiquetas legibles para las entidades.
- `LIMIT`: máximo de registros devueltos.

**Tres transformaciones simples:** cambiar `P17` por `P31` (tipo) o por `P131` (ubicación administrativa); sustituir el sujeto por una variable para invertir el sentido de la búsqueda; aplicar `COUNT(DISTINCT ?propiedad)` para examinar cuántos tipos de propiedades están registrados. Las consultas completas están en `practica/consultas/ejemplos_sparql.md` y en el HTML.

**Advertencia:** el Santuario del TTL local es distinto de la festividad Q2408955. No declaramos que sean una misma entidad.


In [ ]:
# Elige un ejemplo para copiar su SPARQL o abrirlo en el servicio público.
# Este bloque NO realiza llamadas a Internet por sí mismo.
from urllib.parse import quote

EJEMPLOS_SPARQL = {
    "pais": 'SELECT ?pais ?paisLabel WHERE {\n  wd:Q2408955 wdt:P17 ?pais.\n  SERVICE wikibase:label {\n    bd:serviceParam wikibase:language "es,en".\n  }\n}\nLIMIT 10',
    "tipo": 'SELECT ?tipo ?tipoLabel WHERE {\n  wd:Q2408955 wdt:P31 ?tipo.\n  SERVICE wikibase:label {\n    bd:serviceParam wikibase:language "es,en".\n  }\n}\nLIMIT 10',
    "region": 'SELECT ?region ?regionLabel WHERE {\n  wd:Q2408955 wdt:P131 ?region.\n  SERVICE wikibase:label {\n    bd:serviceParam wikibase:language "es,en".\n  }\n}\nLIMIT 10',
    "fecha_registrada": 'SELECT ?fecha WHERE {\n  wd:Q2408955 wdt:P571 ?fecha.\n}\nLIMIT 10',
    "obras_autor": 'SELECT ?obra ?obraLabel WHERE {\n  ?obra wdt:P50 wd:Q35610.\n  SERVICE wikibase:label {\n    bd:serviceParam wikibase:language "es,en".\n  }\n}\nLIMIT 10',
    "conteo_propiedades": 'SELECT (COUNT(DISTINCT ?propiedad) AS ?cantidad) WHERE {\n  wd:Q2408955 ?propiedad ?valor.\n  FILTER(STRSTARTS(\n    STR(?propiedad),\n    "http://www.wikidata.org/prop/direct/"\n  ))\n}',
}

EJEMPLO_A_MOSTRAR = "pais"  # opciones: pais, tipo, region, fecha_registrada, obras_autor, conteo_propiedades
consulta_seleccionada = EJEMPLOS_SPARQL[EJEMPLO_A_MOSTRAR]
print(consulta_seleccionada)
print("\nAbrir en Wikidata:\n", "https://query.wikidata.org/#" + quote(consulta_seleccionada, safe=""))


In [ ]:
CONSULTA_WIKIDATA = '# Wikidata · Qoyllur Rit’i (Q2408955)\n# 4 relaciones de ejemplo, comprobables en el elemento de Wikidata.\n# P17: país; P131: entidad territorial; P31: instancia de.\nSELECT ?relacion ?entidad ?entidadLabel WHERE {\n  VALUES (?relacion ?propiedad) {\n    ("país" wdt:P17)\n    ("división administrativa" wdt:P131)\n    ("tipo" wdt:P31)\n  }\n  wd:Q2408955 ?propiedad ?entidad.\n  SERVICE wikibase:label { bd:serviceParam wikibase:language "es,en". }\n}\nLIMIT 20\n'
print(CONSULTA_WIKIDATA)

RESPALDO_WD = [
 {"relacion":"país","entidad":"Perú","id":"Q419"},
 {"relacion":"división administrativa","entidad":"Departamento del Cusco","id":"Q205057"},
 {"relacion":"tipo","entidad":"festividad religiosa","id":"Q130748394"},
 {"relacion":"tipo","entidad":"peregrinación","id":"Q1644573"},
]

def consultar_wikidata():
    import requests
    try:
        respuesta=requests.get('https://query.wikidata.org/sparql',params={'query':CONSULTA_WIKIDATA,'format':'json'},headers={'Accept':'application/sparql-results+json','User-Agent':'UAH-Taller-Redes/1.0 (academic demonstration)'},timeout=18)
        respuesta.raise_for_status()
        datos=respuesta.json()['results']['bindings']
        resultado=[{'relacion':f['relacion']['value'], 'entidad':f.get('entidadLabel',f['entidad'])['value'], 'id':f['entidad']['value'].rsplit('/',1)[-1]} for f in datos]
        if resultado:
            print('Respuesta EN VIVO de Wikidata (puede cambiar):')
            return pd.DataFrame(resultado)
        print('La consulta no devolvió filas. Se muestra la copia de respaldo.')
    except Exception as err:
        print('Wikidata no respondió; utilizaremos la copia de respaldo. Detalle:',type(err).__name__)
    print('Copia de respaldo, comprobada el 29/10/2026 (NO es consulta en vivo):')
    return pd.DataFrame(RESPALDO_WD)

df_wikidata=consultar_wikidata()
display(df_wikidata)

## 5. Mini-RAG: ¿cómo usamos las relaciones para responder?

**RAG** significa *Retrieval-Augmented Generation* (generación aumentada por recuperación): una pregunta desencadena la búsqueda de información pertinente y esa evidencia se usa como contexto para construir una respuesta.

Aquí mostraremos primero la etapa de **recuperación**, sin un modelo generativo. Convertimos cada relación en una frase con su origen y utilizamos TF–IDF de n-gramas de caracteres para encontrar las más relacionadas con la pregunta.

**Pregunta → relaciones recuperadas → evidencia visible → respuesta opcional.**

Este ejercicio no es un GraphRAG avanzado: no interpreta automáticamente toda la ontología ni realiza razonamiento exhaustivo por múltiples caminos. Su interés metodológico es separar con claridad qué datos existen, cuáles recuperamos y qué respuesta se puede justificar.

**Prueba de control:** si preguntas «¿Quién construyó el Santuario?», estas relaciones no aportan esa información. El sistema debería reconocer el límite y no inventar una atribución.


### Cómo funciona el mini-RAG de este notebook

1. **Leer el archivo:** `rdflib` interpreta el TTL; convertimos las relaciones entidad→entidad a una red dirigida de `networkx`.
2. **Preparar evidencias:** cada arista se convierte en una frase con sujeto, relación, objeto y nombre del archivo de procedencia.
3. **Representar las frases:** `TfidfVectorizer` utiliza secuencias de **2 a 5 caracteres** (`analyzer='char_wb'`); **no** usamos embeddings semánticos.
4. **Ordenar las coincidencias:** calculamos la **similitud coseno** entre pregunta y cada frase; elegimos hasta seis coincidencias por encima de un **umbral didáctico de 0,08** (orientativo, no validado).
5. **Mostrar el contexto:** un resultado como `[T1] Santuario — está en → Sinakara (procedencia: archivo TTL)` es una **referencia interna a una relación**, no una cita bibliográfica.
6. **Responder, si se activa el modelo:** Qwen2.5-0.5B-Instruct recibe las relaciones recuperadas y la instrucción de abstenerse ante evidencias insuficientes. Por defecto no se descarga ni ejecuta.

**Limitaciones:** el buscador es léxico, no recorre caminos de varios saltos, no indexa todas las propiedades literales del TTL y no une automáticamente Wikidata con el grafo local. Una similitud alta puede ser irrelevante para una pregunta concreta; **hay que evaluar si la relación permite contestar**. Las fuentes registradas indican el archivo de procedencia, no necesariamente las fuentes históricas primarias.

**SPARQL vs RAG:** SPARQL busca patrones exactos (si conocemos sus propiedades), mientras que este RAG comienza con una pregunta cotidiana y sugiere relaciones lexicalmente parecidas.


In [ ]:
def inventario_de_relaciones(red):
    filas=[]
    for a,b,d in red.edges(data=True):
        nombre_a=red.nodes[a].get('etiqueta',nombre_corto(a))
        nombre_b=red.nodes[b].get('etiqueta',nombre_corto(b))
        rel=d.get('relacion',d.get('tipo','relacionado con'))
        fuente=d.get('fuente','sin fuente declarada')
        tipo_a=red.nodes[a].get('tipo','')
        tipo_b=red.nodes[b].get('tipo','')
        texto=f'{nombre_a} {rel} {nombre_b}. {tipo_a} {tipo_b}'
        filas.append({'sujeto':nombre_a,'relacion':rel,'objeto':nombre_b,'texto':texto,'fuente':fuente})
    return pd.DataFrame(filas)

def recuperar_relaciones(pregunta, red, k=6, min_similitud=0.08):
    datos=inventario_de_relaciones(red)
    if datos.empty:raise ValueError('El grafo no tiene relaciones para recuperar.')
    vectorizador=TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),strip_accents='unicode')
    matriz=vectorizador.fit_transform(list(datos['texto'])+[pregunta])
    puntuaciones=cosine_similarity(matriz[-1:],matriz[:-1]).ravel()
    seleccion=[int(i) for i in puntuaciones.argsort()[::-1] if puntuaciones[i] >= min_similitud][:min(k,len(datos))]
    salida=datos.iloc[seleccion].copy().reset_index(drop=True)
    salida.insert(0,'evidencia',[f'T{i+1}' for i in range(len(salida))])
    salida.insert(1,'similitud',[round(float(puntuaciones[i]),3) for i in seleccion])
    return salida

PREGUNTA = '¿Dónde se encuentra el Santuario del Señor de Qoyllur Rit’i?'
evidencias=recuperar_relaciones(PREGUNTA,G,k=6)
display(evidencias[['evidencia','similitud','sujeto','relacion','objeto','fuente']])

# Pregunta adicional, pensada para reconocer los límites:
# PREGUNTA = '¿Quién diseñó el santuario y en qué año?'
# evidencias = recuperar_relaciones(PREGUNTA,G,k=6)

In [ ]:
def contexto_del_grafo(pregunta,red,k=6):
    tabla=recuperar_relaciones(pregunta,red,k)
    partes=[f"[{fila.evidencia}] {fila.sujeto} —{fila.relacion}→ {fila.objeto} (procedencia: {fila.fuente})" for fila in tabla.itertuples()]
    return "\n".join(partes) if partes else "NO HAY RELACIONES CON COINCIDENCIA LÉXICA; no es posible fundamentar una respuesta."

def preparar_prompt(pregunta,red,k=6):
    return f"""Responde en español solo con las relaciones documentadas aquí.
No uses conocimiento externo. Toda afirmación factual debe citar [T1], [T2], etc.
Si la evidencia es insuficiente para responder, dilo de forma explícita.
No supongas que una entidad local y una entidad de Wikidata son la misma.

PREGUNTA: {pregunta}

RELACIONES RECUPERADAS:
{contexto_del_grafo(pregunta,red,k)}

RESPUESTA:""" 

PROMPT=preparar_prompt(PREGUNTA,G)
print(PROMPT)
print('\n--- FIN DEL CONTEXTO: comprobar los triples originales antes de interpretar. ---')

## 6. Generación opcional con un LLM abierto (sin clave API)
La etapa anterior ya realiza una **recuperación real de conocimiento estructurado**, dejando visible qué relaciones se usan. Para mostrar la **G** de RAG, podemos añadir un modelo generativo pequeño.

El bloque siguiente descarga **Qwen2.5-0.5B-Instruct** desde Hugging Face (aproximadamente 1 GB de pesos). Puede tardar o requerir memoria; por eso la ejecución es **opcional** y viene desactivada. No se necesita cuenta de pago. La salida generada debe cotejarse con las evidencias, incluso si trae identificadores.

**En el taller:** activar la celda únicamente cuando la recuperación esté demostrada; alternativamente, copiar el prompt anterior y usar un LLM disponible para mostrar la fase generativa. Un sistema local con 0,5B parámetros no garantiza respuestas rigurosas en español.

In [ ]:
GENERAR_CON_MODELO_ABIERTO = False  # Cambiar a True para una demostración de generación.

if GENERAR_CON_MODELO_ABIERTO:
    import subprocess, sys
    subprocess.check_call([sys.executable,'-m','pip','-q','install','transformers>=4.44,<5','accelerate>=0.26'])
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    modelo_id='Qwen/Qwen2.5-0.5B-Instruct'
    tokenizer=AutoTokenizer.from_pretrained(modelo_id)
    modelo=AutoModelForCausalLM.from_pretrained(modelo_id,torch_dtype='auto',device_map='auto')
    mensajes=[
       {'role':'system','content':'Responde solamente a partir de las evidencias entregadas. Si no hay evidencia suficiente, dilo. Cita [T1], [T2], etc.'},
       {'role':'user','content':PROMPT},
    ]
    entradas=tokenizer.apply_chat_template(mensajes,tokenize=True,add_generation_prompt=True,return_tensors='pt').to(modelo.device)
    with torch.no_grad():
       generados=modelo.generate(entradas,max_new_tokens=180,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(generados[0][entradas.shape[1]:],skip_special_tokens=True))
    print('\nRevisa cada afirmación contra las relaciones recuperadas; la generación puede contener errores.')
else:
    print('Modelo generativo desactivado. La recuperación del grafo funciona y el prompt fundamentado ya está preparado.')

## 7. Qué significa el resultado
1. **Un nodo central no es necesariamente la entidad culturalmente más importante.** Su puntuación depende de las relaciones que codificamos.
2. **Una arista no es automáticamente una evidencia independiente.** Hay que conocer su fuente y el criterio de incorporación.
3. **Un grafo local y Wikidata no son intercambiables.** El primero incluye observaciones situadas; el segundo se construye colaborativamente con sus propias reglas de representación.
4. **Un RAG no corrige una fuente mal digitalizada ni recupera relaciones ausentes.** Su respuesta depende de los datos de entrada y del sistema de recuperación.

**Para continuar:** sustituye el archivo de ejemplo con tu propio TTL o tu CSV de relaciones, revisa sus tipos y procede con las mismas celdas. Para un sistema de investigación, se necesitarían controles adicionales: equivalencia de entidades, trazabilidad por triple, validación con expertos, límites de acceso y evaluación sistemática de respuestas.

### Fuentes y créditos
- TTL del proyecto sobre Qoyllur Rit’i: selección de relaciones aportada por los autores para la práctica, no publicación íntegra del corpus.
- [Wikidata Q2408955](https://www.wikidata.org/wiki/Q2408955) · [Servicio SPARQL](https://query.wikidata.org/).
- Concepto de RAG: recuperación de evidencias → construcción de contexto → respuesta generativa. La generación se presenta como etapa opcional.
- Taller y materiales: Javier Vera Zúñiga y Dina Cornejo Meza. Universidad Alberto Hurtado, 2026.